# Contextual absractions

## Contextual abstractions:: introduction

Contextual abstractions are programming ways to provide context-dependent constructions.
These can take different forms and be used in different ways, for example to inject context values, context-dependent parameters in functions, or to provide contextual conversions among types.
Typical examples include:

- Establishing context variables
- Type classes
- Dependency injection
- Expressing capabilities
- Computing new types


## Contextual abstractions in Scala

In Scala contextual abstractions existed for a long time, mostly through `implicit` values and methods in Scala 2. 

In **Scala 3** the entire desing has been revisited, and it now provides a unified paradigm for:

 - Retroactively **extending** classes: `extension` methods
 - Abstracting over **contextual** information: `using` clauses
 - **Type-class** instances: `given` instances for the canonical value of a certain type. 
 - Viewing one type as another: **implicit** conversion
 - **Higher-order** contextual abstractions
 - Actionable feedback from the compiler: `import` suggestions

## Extension methods

This first type of abstraction allows to **add methods** to a type *after* the type is defined

For instance consider that this class representing a `Student` already exists in the code:

In [14]:
case class Student(name:String, age: Int)

defined class Student

Now we want to add a method to check if the student is old enough to enrol.

A siple way would be to create a helper method in a companion object:

In [2]:
object StudentExtensions :
    def isOldEnough(s:Student) = s.age > 18

defined object StudentExtensions

However this is a bit cumbersome, it would be convenient to call the method directly from the student. We can do that using the `extension` keyword adn define the new method over the `Student` class:

In [15]:
extension (s:Student)
  def isOldEnough = s.age > 18

defined extension methods 

Now if we have an instance of a student, we can simply invoke the new method:

In [16]:
val jim = Student("jim",17)
jim.isOldEnough


jim: Student = Student(name = "jim", age = 17)
res16_1: Boolean = false

In fact multiple methods can be defined in this manner:

In [17]:
extension (s:Student)
  def isOldEnough = s.age > 18
  def beFriend(s2:Student) = (s,"is friend of",s2)

defined extension methods 

In [8]:
jim.beFriend(Student("jill",21))

res8: (Student, String, Student) = (
  Student(name = "jim", age = 17),
  "is friend of",
  Student(name = "jill", age = 21)
)

Extensions can also include working with other traits, classes, etc. as any other method:

In [18]:
trait University:
    val name:String

defined trait University

In [19]:
extension (s:Student)
  def applyToUniversity(u:University)=
    println(s"$s.name applied to $u")
    u.name

defined extension methods 

## Contextual values with `using`

Introducing values with the `using` keyword allows bringing **context information** to the different components of your system (e.g., config, settings)


For example consider this method that allows a student to resign from a `University`:

In [22]:
case class Student(name:String, age: Int):
  def quit(reason:String,u:University) =
    s"good reason to leave ${u.name}"


defined class Student

A student Jim can resign to a university, as other students can, too:

In [24]:
val hesso = new University{val name="HES-SO"}
val jim =Student("jim",17)
jim.quit("tired",hesso)

hesso: University = ammonite.$sess.cmd24$$anon$1@242d79fe
jim: Student = Student(name = "jim", age = 17)
res24_2: String = "good reason to leave HES-SO"

In [25]:
Student("joe",18).quit("bored",hesso)
Student("moe",20).quit("sad",hesso)
Student("doe",21).quit("exahusted",hesso)
Student("roe",17).quit("bored",hesso)

res25_0: String = "good reason to leave HES-SO"
res25_1: String = "good reason to leave HES-SO"
res25_2: String = "good reason to leave HES-SO"
res25_3: String = "good reason to leave HES-SO"

However if all student in our code are from the same university, it is kind of repetitive to always specify the same university.
In fact this is really part of the **context** of the application. 

We can then separate the university parameter and indicate that it is contestual with the `using` keyword:

In [29]:
case class Student(name:String, age: Int):
  def quit(reason:String)(using u:University) =
    s"good reason to leave ${u.name}"


defined class Student

Now, through the `given` keyword we can set the `University` that we want to put into the context:


In [30]:
val jim =Student("jim",17)

jim: Student = Student(name = "jim", age = 17)

In [31]:
given University = hesso 

given_University: University = <given>

And call the quit method knowing that *implicitly* will use the contextual university:

In [33]:
jim.quit("tired")

res33: String = "good reason to leave HES-SO"

We can still call the method with the university in an explicit way:

In [37]:
jim.quit("tired")(using hesso)

res37: String = "good reason to leave HES-SO"

Beware that context can be set again to other values, depending on what is required:

In [35]:
given University = new University{val name="EPFL"}

given_University: University = <given>

In [36]:
jim.quit("tired")

res36: String = "good reason to leave EPFL"

## Given imports

With contextual abstractions, it is possible to import them from different libraries and packages.

It is possible to provide a clear scope for this purpose, so that we can know where any `given` values come from

For example, consider that there is a certain `Geography` object that defines a `given` instance of a `Country`:

In [38]:
trait Country:
  val code:String

defined trait Country

In [39]:
object Geography:
  given Country = 
    new Country{val code="CH"}

defined object Geography

Then this country could be used somewhere else in the code, for example for this `obtainDiploma` method:

In [40]:
case class Student(name:String, age: Int):
  def obtainDiploma(using c:Country) =
    println(s"diploma in Country ${c.code}")

defined class Student

In [41]:
val jim = Student("jim",17)

jim: Student = Student(name = "jim", age = 17)

However, to use it we have to make the compiler know about it, otherwise it is not in the scope of this code.
The compiler can give some suggestions, though:

In [41]:
jim.obtainDiploma

-- [E172] Type Error: cmd42.sc:1:29 --------------------------------------------
1 |val res42 = jim.obtainDiploma
  |                             ^
  |No given instance of type ammonite.$sess.cmd40.wrapper.cmd38.Country was found for parameter c of method obtainDiploma in class Student
  |
  |The following import might fix the problem:
  |
  |  import cmd42.this.cmd39.Geography.given_Country
  |
Compilation Failed

A simple import does the trick:

In [19]:
import Geography.given

import Geography.given


In [20]:
jim.obtainDiploma

diploma in Country CH


Imports can be specific, indicating what parts of the object (or package) should be imported.

In [21]:
import Geography.{given,*}

import Geography.{given,*}


## Type classes

Type claases work as a parameterized type, which adds new behavior to any closed data type without using sub-typing.
It can be handy for:

 - expressing how a type you don’t own conforms to such behavior
 - expressing behavior for multiple types without sub-typing relationships


For example consider this `Architect` class:

In [43]:
case class Architect(name: String, income:Double)

defined class Architect

Now let's say we want to compute taxes for architects, and also for other types of professionals. 
We can create a **type class** for this:

In [46]:
trait Taxable[A]:
  extension(a:A) 
    def computeTax:Double

defined trait Taxable

To use it, we can use a contextual `given` for the `Architect` class. This means creating extension methods for it:

In [48]:
given Taxable[Architect] with
  extension(a:Architect) 
    def computeTax = a.income * 0.3

defined object 

We can do the same for other classes, as for the `Student`. Again we canimplement the `computeTax` method.

In [50]:
given Taxable[Student] with
  extension(s:Student)
    def computeTax = if s.age > 18 then 1000 else 50

defined object 

Now to use it, we can simply instantiate an architect:

In [52]:
val dude = Architect("Dude",2000)
dude.computeTax

dude: Architect = Architect(name = "Dude", income = 2000.0)
res52_1: Double = 600.0

And same for a Student:

In [54]:
val jim = Student("jim",17)
jim.computeTax

jim: Student = Student(name = "jim", age = 17)
res54_1: Double = 50.0

Further we can use the `Taxable` type class in an arbitrary method:

In [56]:
def computeFullTax[T:Taxable](somebody:T)=
  val fixed = 250
  somebody.computeTax + fixed 

defined function computeFullTax

And then just use it with concrete instances:

In [57]:
val tim = jim.copy(name = "tim")

tim: Student = Student(name = "tim", age = 17)

In [58]:
computeFullTax(dude)

res58: Double = 850.0

In [60]:
computeFullTax(tim)

res60: Double = 300.0

## Implicit conversions

This abstraction allows converting form one type to another seamlessly.

Conversiond are defined by `given` instances of the `scala.Conversion` class.


The conversion is defined through the `apply` method on a `Conversion` instance.
For example, this one convers a Student into an Architect:

In [61]:
given Conversion[Student,Architect] with
  def apply(s:Student) = Architect(s.name,s.age*4000)

defined object 

Then if thre is a method that requires an `Architect`, they can pass a student instead.

In [62]:
def hireArchitect(a:Architect,offer:Double) =
  offer>a.income

defined function hireArchitect

In [63]:
val jim = Student("jim",17)

jim: Student = Student(name = "jim", age = 17)

In [65]:
hireArchitect(jim,800)

-- Warning: cmd65.sc:1:26 ------------------------------------------------------
1 |val res65 = hireArchitect(jim,800)
  |                          ^^^
  |Use of implicit conversion object given_Conversion_Student_Architect in class Helper should be enabled
  |by adding the import clause 'import scala.language.implicitConversions'
  |or by setting the compiler option -language:implicitConversions.
  |See the Scala docs for value scala.language.implicitConversions for a discussion
  |why the feature should be explicitly enabled.


res65: Boolean = false

You see that a warning is issued, suggesting to import the implicit conversions:

In [66]:
import scala.language.implicitConversions

import scala.language.implicitConversions


In [67]:
hireArchitect(jim,800)

res67: Boolean = false

## Summoning classes

In some occassions we can *summon*, or call for instances of a certain Class or Trait that are in the context.

For example consider a `GradeCalculator` trait, which represents an abstraction for different ways of calculating grades.

In [40]:
case class Grade(value:Int)

trait GradeCalculator:
  def assessGrade(g: Grade): Option[String]

defined class Grade
defined trait GradeCalculator

Then we can define a `given` instance of a specific calculator that implements this trait:

In [28]:
given defaultGradeCalculator: GradeCalculator with
  def assessGrade(g: Grade): Option[String] =
    if g.value < 4 then Some("Bad")
    else if g.value < 5 then Some("So so")
    else if g.value <= 6 then Some("Great")
    else None

defined class Grade
defined trait GradeCalculator
defined object defaultGradeCalculator

Now if in a method we want to use the calculator we can `summon` it. Notice that it will take whatever `GradeCalculator` it finds in the current context.

In [29]:
def processGrade(g: Grade) =
  val calc = summon[GradeCalculator]
  val level = calc.assessGrade(g)
  println(s"level: $level")

defined function processGrade

In [40]:
val grade = Grade(3)
processGrade(grade)

-- [E007] Type Mismatch Error: cmd41.sc:2:27 -----------------------------------
2 |val res41_1 = processGrade(grade)
  |                           ^^^^^
  |      Found:    (Helper.this.grade : cmd41.this.cmd40.Grade)
  |      Required: ammonite.$sess.cmd36.wrapper.cmd28.Grade²
  |
  |      where:    Grade  is a class in class Helper²
  |                Grade² is a class in class Helper³
  |
  |
  |      The following import might make progress towards fixing the problem:
  |
  |        import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

We can have other implementations, for example this *strict* calculator

In [35]:
given strictGrading: GradeCalculator with
  def assessGrade(g: Grade): Option[String] =
    if g.value <= 5 then Some("Terrible")
    else if g.value <= 6 then Some("Fine")
    else None

defined object strictGrading

However we see that now there are two different calculators in the context so the compiler will complain:

In [35]:
def processGrade(g: Grade) =
  val calc = summon[GradeCalculator]
  val level = calc.assessGrade(g)
  println(s"level: $level")

-- [E172] Type Error: cmd36.sc:2:36 --------------------------------------------
2 |  val calc = summon[GradeCalculator]
  |                                    ^
  |Ambiguous given instances: both object strictGrading in class Helper and given instance gc in class Helper match type cmd36.this.cmd28.GradeCalculator of parameter x of method summon in object Predef
Compilation Failed

In case of ambiguity, it is alwas possible to be explicit on which `GradeCalculator` we want to put in the current context, with a `given`:

In [36]:
given gc:GradeCalculator = strictGrading

def processGrade(g: Grade) =
  val calc = summon[GradeCalculator]
  val level = calc.assessGrade(g)
  println(s"level: $level")

gc: GradeCalculator = <given>
defined function processGrade

In [39]:
processGrade(Grade(5))

level: Some(Terrible)
